In [ ]:
import os

base_path = "/home/siu856622573/Code/RQ_2_3/top_pos10"

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".json"):
            print(os.path.join(root, file))

### Accuracy Check

In [4]:
import json
import os

BASE_PATH = "/home/siu856622573/Code/RQ_2_3/top_pos10"

VALID_LABELS = {"ADHD", "Control"}

# Model name -> filename prefix
MODELS = {
    "DeepSeek": "deepseek",
    "Llama-3.1-8B": "llama",
    "Qwen2.5-14B": "qwen",
    "Qwen3-14B": "qwen3:14b",
    "GPT-5.6-LUNA": "gpt_luna",
    "DeepSeek-V4-Flash": "deepseek_v4",
}

# Scenario -> filename
SCENARIO_FILES = {
    1: "_pos10_fc_scn1.json",
    2: "_pos10_fc_scn2_named_prompts.json",
    3: "_pos10_fc_named_scn3.json",
    4: "_pos10_zscore_scn4.json",
    5: "_pos10_zscore_named_prompt_scn5.json",
    6: "_pos10_named_zscored_scn6.json",
}


def calculate_metrics(filepath):
    with open(filepath, "r") as f:
        data = json.load(f)

    # Only subjects with valid prediction AND valid ground truth
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS
        and r.get("prediction") in VALID_LABELS
    ]

    if not scorable:
        return None, None, None

    # Overall Accuracy (A)
    correct = sum(
        1 for r in scorable
        if r["prediction"] == r["ground_truth"]
    )

    accuracy = correct / len(scorable)

    # ADHD performance (H)
    adhd = [
        r for r in scorable
        if r["ground_truth"] == "ADHD"
    ]

    adhd_correct = sum(
        1 for r in adhd
        if r["prediction"] == "ADHD"
    )

    adhd_accuracy = (
        adhd_correct / len(adhd)
        if adhd else None
    )

    # Control performance (C)
    control = [
        r for r in scorable
        if r["ground_truth"] == "Control"
    ]

    control_correct = sum(
        1 for r in control
        if r["prediction"] == "Control"
    )

    control_accuracy = (
        control_correct / len(control)
        if control else None
    )

    return accuracy, adhd_accuracy, control_accuracy


# --------------------------------------------------
# Find all JSON files recursively
# --------------------------------------------------

all_files = {}

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        if file.endswith(".json"):
            all_files[file] = os.path.join(root, file)


# --------------------------------------------------
# Build results
# --------------------------------------------------

results = {}

for scenario, suffix in SCENARIO_FILES.items():

    results[scenario] = {}

    for model, prefix in MODELS.items():

        filename = prefix + suffix

        if filename not in all_files:
            print(f"WARNING: File not found: {filename}")
            results[scenario][model] = (None, None, None)
            continue

        filepath = all_files[filename]

        results[scenario][model] = calculate_metrics(filepath)


# --------------------------------------------------
# Print table
# --------------------------------------------------

print("\n" + "=" * 130)
print("SCENARIO PERFORMANCE")
print("=" * 130)

header = (
    f"{'Scenario':<10}|"
    f"{'R1-Distill':<25}|"
    f"{'Llama':<25}|"
    f"{'Qwen2.5':<25}|"
    f"{'Qwen3':<25}|"
    f"{'LUNA':<25}"
    f"{'DeepSeek-V4':<25}"
)

print(header)
print("-" * 110)

for scenario in range(1, 7):

    row = f"{'Scn ' + str(scenario):<10}|"

    for i, model in enumerate(MODELS):

        metrics = results[scenario][model]

        if metrics[0] is None:
            value = "N/A"
        else:
            A, D, C = metrics
            value = f"A={A:.2%}, D={D:.2%}, C={C:.2%}"

        row += f"{value:<25}"

        # Vertical line after every model except the last one
        if i < len(MODELS) - 1:
            row += "|"

    print(row)

print("=" * 130)
print("A = Overall Accuracy | D = ADHD Accuracy | C = Control Accuracy")
print("=" * 130)


SCENARIO PERFORMANCE
Scenario  |R1-Distill               |Llama                    |Qwen2.5                  |Qwen3                    |LUNA                     DeepSeek-V4              
--------------------------------------------------------------------------------------------------------------
Scn 1     |A=51.20%, D=51.09%, C=51.27%|A=59.57%, D=19.05%, C=82.67%|A=63.28%, D=0.37%, C=99.79%|A=52.93%, D=43.73%, C=58.20%|A=55.41%, D=36.07%, C=66.53%|A=52.81%, D=33.57%, C=63.92%
Scn 2     |A=53.15%, D=45.56%, C=57.44%|A=58.20%, D=13.45%, C=83.98%|A=63.71%, D=1.10%, C=99.17%|A=41.05%, D=80.43%, C=18.60%|A=45.89%, D=53.21%, C=41.68%|A=57.81%, D=18.21%, C=80.53%
Scn 3     |A=50.69%, D=28.52%, C=63.40%|A=50.68%, D=52.59%, C=49.57%|A=63.27%, D=0.00%, C=100.00%|A=50.40%, D=56.04%, C=47.12%|A=54.04%, D=33.57%, C=65.78%|A=55.60%, D=26.07%, C=72.54%
Scn 4     |A=44.11%, D=78.91%, C=24.17%|A=36.86%, D=92.83%, C=4.73%|A=60.44%, D=7.53%, C=90.76%|A=40.13%, D=83.46%, C=15.22%|A=50.00%, D=42.31%, C=5

# McNemar's test

In [21]:
import json
from statsmodels.stats.contingency_tables import mcnemar
import pandas as pd

VALID_LABELS = {"ADHD", "Control"}

BASE_DIR = "/home/siu856622573/Code/RQ_2_3/top_pos10"

MODEL_DIRS = {
    "DeepSeek-R1-Distill-14B": f"{BASE_DIR}/deepseek-r1-distill-qwen-14b",
    "Llama-3.1-8B": f"{BASE_DIR}/llama-3.1-8B-Instruct",
    "Qwen2.5-14B": f"{BASE_DIR}/qwen2.5-14B-Instruct",
    "Qwen3-14B": f"{BASE_DIR}/qwen3-14b",
    "DeepSeek-V4": f"{BASE_DIR}/deepseek_v4",
    "GPT-LUNA": f"{BASE_DIR}/gpt_luna",
}

def load(path):
    with open(path) as f:
        return json.load(f)

def flip_and_mcnemar(file_a, file_b, gt):
    a, b = load(file_a), load(file_b)
    common = set(a) & set(b) & set(gt)

    flips = b_count = c_count = skipped = 0
    for sid in common:
        entry_a, entry_b = a[sid], b[sid]
        pred_a, pred_b = entry_a.get("prediction"), entry_b.get("prediction")
        if pred_a not in VALID_LABELS or pred_b not in VALID_LABELS:
            skipped += 1
            continue
        if pred_a != pred_b:
            flips += 1
        correct_a, correct_b = (pred_a == gt[sid]), (pred_b == gt[sid])
        if correct_a and not correct_b:
            b_count += 1
        elif not correct_a and correct_b:
            c_count += 1

    scorable = len(common) - skipped
    flip_rate = flips / scorable if scorable > 0 else float("nan")
    p = mcnemar([[0, b_count], [c_count, 0]], exact=True).pvalue
    return flip_rate, b_count, c_count, p, scorable, skipped

# ============================================================
# Ground truth
# ============================================================
def load_ground_truth(csv_path):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "ADHD"}  # DX_binary
    return {str(row["subject_id"]): label_map[int(row["DX_binary"])] for _, row in df.iterrows()}

gt = load_ground_truth("/home/siu856622573/Code/subject_summary.csv")  # from earlier in this conversation

MODEL_FILES = {
    "DeepSeek-R1-Distill-14B": {
        "scn1": "deepseek_pos10_fc_scn1.json",
        "scn2": "deepseek_pos10_fc_scn2_named_prompts.json",
        "scn3": "deepseek_pos10_fc_named_scn3.json",
        "scn4_content": "deepseek_pos10_zscore_named_prompt_scn5.json",  # names + z
        "scn5_content": "deepseek_pos10_zscore_scn4.json",               # no names + z
        "scn6": "deepseek_pos10_named_zscored_scn6.json",
    },
    "Llama-3.1-8B": {
        "scn1": "llama_pos10_fc_scn1.json",
        "scn2": "llama_pos10_fc_scn2_named_prompts.json",
        "scn3": "llama_pos10_fc_named_scn3.json",
        "scn4_content": "llama_pos10_zscore_named_prompt_scn5.json",
        "scn5_content": "llama_pos10_zscore_scn4.json",
        "scn6": "llama_pos10_named_zscored_scn6.json",
    },
    "Qwen2.5-14B": {
        "scn1": "qwen_pos10_fc_scn1.json",
        "scn2": "qwen_pos10_fc_scn2_named_prompts.json",
        "scn3": "qwen_pos10_fc_named_scn3.json",
        "scn4_content": "qwen_pos10_zscore_named_prompt_scn5.json",
        "scn5_content": "qwen_pos10_zscore_scn4.json",
        "scn6": "qwen_pos10_named_zscored_scn6.json",
    },
    "Qwen3-14B": {
        "scn1": "qwen3:14b_pos10_fc_scn1.json",
        "scn2": "qwen3:14b_pos10_fc_scn2_named_prompts.json",
        "scn3": "qwen3:14b_pos10_fc_named_scn3.json",
        "scn4_content": "qwen3:14b_pos10_zscore_named_prompt_scn5.json",
        "scn5_content": "qwen3:14b_pos10_zscore_scn4.json",
        "scn6": "qwen3:14b_pos10_named_zscored_scn6.json",
    },
    "DeepSeek-V4": {
        "scn1": "deepseek_v4_pos10_fc_scn1.json",
        "scn2": "deepseek_v4_pos10_fc_scn2_named_prompts.json",
        "scn3": "deepseek_v4_pos10_fc_named_scn3.json",
        "scn4_content": "deepseek_v4_pos10_zscore_named_prompt_scn5.json",
        "scn5_content": "deepseek_v4_pos10_zscore_scn4.json",
        "scn6": "deepseek_v4_pos10_named_zscored_scn6.json",
    },
    "GPT-LUNA": {
        "scn1": "gpt_luna_pos10_fc_scn1.json",
        "scn2": "gpt_luna_pos10_fc_scn2_named_prompts.json",
        "scn3": "gpt_luna_pos10_fc_named_scn3.json",
        "scn4_content": "gpt_luna_pos10_zscore_named_prompt_scn5.json",
        "scn5_content": "gpt_luna_pos10_zscore_scn4.json",
        "scn6": "gpt_luna_pos10_named_zscored_scn6.json",
    },
}

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def flip_and_mcnemar(file_a, file_b, gt):
    a, b = load_json(file_a), load_json(file_b)
    common = set(a) & set(b) & set(gt)
    flips = b_count = c_count = skipped = 0
    for sid in common:
        pred_a, pred_b = a[sid].get("prediction"), b[sid].get("prediction")
        if pred_a not in VALID_LABELS or pred_b not in VALID_LABELS:
            skipped += 1
            continue
        if pred_a != pred_b:
            flips += 1
        correct_a, correct_b = (pred_a == gt[sid]), (pred_b == gt[sid])
        if correct_a and not correct_b:
            b_count += 1
        elif not correct_a and correct_b:
            c_count += 1

    scorable = len(common) - skipped
    flip_rate = flips / scorable if scorable > 0 else float("nan")
    p = mcnemar([[0, b_count], [c_count, 0]], exact=True).pvalue
    return flip_rate, b_count, c_count, p, scorable, skipped
# ============================================================
# Run RQ2 (naming) + RQ3 (z-scoring) for all 6 models
# ============================================================
import os

rows = []
for model, f in MODEL_FILES.items():
    d = MODEL_DIRS[model]

    fr, b, c, p, scorable, skipped = flip_and_mcnemar(
        os.path.join(d, f["scn1"]), os.path.join(d, f["scn2"]), gt
    )
    rows.append({"Model": model, "RQ": "RQ2", "Comparison": "No names vs Names",
                 "Flip Rate": round(fr, 3), "b": b, "c": c, "p": round(p, 4),
                 "Scorable": scorable, "Skipped": skipped})

    fr, b, c, p, scorable, skipped = flip_and_mcnemar(
        os.path.join(d, f["scn1"]), os.path.join(d, f["scn5_content"]), gt
    )
    rows.append({"Model": model, "RQ": "RQ3", "Comparison": "No names: raw vs z",
                 "Flip Rate": round(fr, 3), "b": b, "c": c, "p": round(p, 4),
                 "Scorable": scorable, "Skipped": skipped})

    fr, b, c, p, scorable, skipped = flip_and_mcnemar(
        os.path.join(d, f["scn2"]), os.path.join(d, f["scn4_content"]), gt
    )
    rows.append({"Model": model, "RQ": "RQ3", "Comparison": "Names: raw vs z",
                 "Flip Rate": round(fr, 3), "b": b, "c": c, "p": round(p, 4),
                 "Scorable": scorable, "Skipped": skipped})

    fr, b, c, p, scorable, skipped = flip_and_mcnemar(
        os.path.join(d, f["scn3"]), os.path.join(d, f["scn6"]), gt
    )
    rows.append({"Model": model, "RQ": "RQ3", "Comparison": "Relationship: raw vs z",
                 "Flip Rate": round(fr, 3), "b": b, "c": c, "p": round(p, 4),
                 "Scorable": scorable, "Skipped": skipped})

results_df = pd.DataFrame(rows)
results_df.to_csv("rq2_rq3_mcnemar_all_models.csv", index=False)

for model in MODEL_FILES:
    print("=" * 100)
    print(model)
    print("=" * 100)
    display(results_df[results_df["Model"] == model].drop(columns="Model"))

DeepSeek-R1-Distill-14B


,RQ,Comparison,Flip Rate,b,c,p,Scorable,Skipped
0,RQ2,No names vs Names,0.526,185,199,0.5071,730,38
1,RQ3,No names: raw vs z,0.518,219,163,0.0048,737,31
2,RQ3,Names: raw vs z,0.541,214,152,0.0014,677,91
3,RQ3,Relationship: raw vs z,0.570,243,161,0.0001,709,59


Llama-3.1-8B


,RQ,Comparison,Flip Rate,b,c,p,Scorable,Skipped
4,RQ2,No names vs Names,0.270,82,79,0.8748,597,171
5,RQ3,No names: raw vs z,0.776,376,205,0.0000,749,19
6,RQ3,Names: raw vs z,0.557,158,106,0.0016,474,294
7,RQ3,Relationship: raw vs z,0.504,210,160,0.0108,734,34


Qwen2.5-14B


...

,RQ,Comparison,Flip Rate,b,c,p,Scorable,Skipped
12,RQ2,No names vs Names,0.548,254,162,0.0000,759,9
13,RQ3,No names: raw vs z,0.547,253,154,0.0000,744,24
14,RQ3,Names: raw vs z,0.237,94,76,0.1921,717,51
15,RQ3,Relationship: raw vs z,0.487,160,111,0.0035,556,212


DeepSeek-V4


,RQ,Comparison,Flip Rate,b,c,p,Scorable,Skipped
16,RQ2,No names vs Names,0.404,136,173,0.0404,765,3
17,RQ3,No names: raw vs z,0.599,277,181,0.0000,764,4
18,RQ3,Names: raw vs z,0.636,297,191,0.0000,767,1
19,RQ3,Relationship: raw vs z,0.605,284,181,0.0000,768,0


GPT-LUNA


,RQ,Comparison,Flip Rate,b,c,p,Scorable,Skipped
20,RQ2,No names vs Names,0.538,242,170,0.0005,766,2
21,RQ3,No names: raw vs z,0.445,173,140,0.0703,704,64
22,RQ3,Names: raw vs z,0.425,187,132,0.0024,750,18
23,RQ3,Relationship: raw vs z,0.642,310,182,0.0000,766,2
